# Fine-Tune XLM-RoBERTa NER on Gold Set

Transfer learning from `Davlan/xlm-roberta-base-ner-hrl` to detect 5 PII types:
NAME, PHONE, EMAIL, ADDRESS, ID across EN/ZH/mixed text.

Anti-overfitting strategy for ~120 records:
- 5-fold cross-validation
- Early stopping (patience=3)
- Weight decay + dropout
- Data augmentation via entity substitution

In [2]:
import json
import gc
import os
import time
from pathlib import Path

import numpy as np
from datasets import Dataset, Features, Sequence, Value
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
import torch

# Detect device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    DEVICE = "mps"
    USE_FP16 = False  # MPS doesn't support fp16 training
    print("Using Apple MPS (Metal) GPU")
elif torch.cuda.is_available():
    DEVICE = "cuda"
    USE_FP16 = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    USE_FP16 = False
    print("Using CPU (this will be slow)")

# Config
BASE_MODEL = "Davlan/xlm-roberta-base-ner-hrl"
DATA_DIR = Path("data/prepared/gold_bio")
OUTPUT_DIR = Path("data/models/ner_finetuned_gold_v1")
NUM_FOLDS = 5

# Path resolution: handle running from notebooks/ or project root
if not DATA_DIR.exists() and (Path("..") / DATA_DIR).exists():
    DATA_DIR = Path("..") / DATA_DIR
    OUTPUT_DIR = Path("..") / OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")

LABEL_LIST = [
    "O",
    "B-NAME", "I-NAME",
    "B-PHONE", "I-PHONE",
    "B-EMAIL", "I-EMAIL",
    "B-ADDRESS", "I-ADDRESS",
    "B-ID", "I-ID",
]
LABEL_TO_ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: l for l, i in LABEL_TO_ID.items()}
print(f"Labels: {len(LABEL_LIST)} ({LABEL_LIST})")

/Users/jiarui/Documents/LLM/AAI3008-multilingual-PII-masking/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jiarui/Documents/LLM/AAI3008-multilingual-PII-masking/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Apple MPS (Metal) GPU
DATA_DIR: /Users/jiarui/Documents/LLM/AAI3008-multilingual-PII-masking/data/prepared/gold_bio
OUTPUT_DIR: /Users/jiarui/Documents/LLM/AAI3008-multilingual-PII-masking/data/models/ner_finetuned_gold_v1
Labels: 11 (['O', 'B-NAME', 'I-NAME', 'B-PHONE', 'I-PHONE', 'B-EMAIL', 'I-EMAIL', 'B-ADDRESS', 'I-ADDRESS', 'B-ID', 'I-ID'])


In [3]:
def load_bio_jsonl(path: Path) -> Dataset:
    """Load BIO JSONL into HuggingFace Dataset."""
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return Dataset.from_dict({
        "input_ids": [r["input_ids"] for r in records],
        "attention_mask": [r["attention_mask"] for r in records],
        "labels": [r["labels"] for r in records],
    })

In [4]:
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report


def compute_metrics(eval_pred):
    """Compute entity-level P/R/F1 using seqeval."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(predictions, labels):
        t_labels = []
        t_preds = []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            t_labels.append(ID_TO_LABEL[l])
            t_preds.append(ID_TO_LABEL.get(p, "O"))
        true_labels.append(t_labels)
        true_preds.append(t_preds)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

In [2]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

NameError: name 'AutoTokenizer' is not defined

## 5-Fold Cross-Validation Training

In [ ]:
fold_results = []
cv_start = time.time()

for fold in range(NUM_FOLDS):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    fold_dir = DATA_DIR / f"fold_{fold}"
    train_ds = load_bio_jsonl(fold_dir / "train.jsonl")
    val_ds = load_bio_jsonl(fold_dir / "val.jsonl")

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

    model = AutoModelForTokenClassification.from_pretrained(
        BASE_MODEL,
        num_labels=len(LABEL_LIST),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        ignore_mismatched_sizes=True,
    )

    fold_output = OUTPUT_DIR / f"fold_{fold}"

    training_args = TrainingArguments(
        output_dir=str(fold_output),
        learning_rate=2e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,  # effective batch = 4
        num_train_epochs=15,
        weight_decay=0.01,
        warmup_ratio=0.1,
        fp16=USE_FP16,
        use_mps_device=(DEVICE == "mps"),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=10,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    fold_start = time.time()
    trainer.train()
    fold_elapsed = time.time() - fold_start

    # Evaluate
    metrics = trainer.evaluate()
    metrics["fold"] = fold
    metrics["train_time_sec"] = fold_elapsed
    metrics["best_epoch"] = trainer.state.best_model_checkpoint.split("-")[-1] if trainer.state.best_model_checkpoint else None
    fold_results.append(metrics)
    print(f"Fold {fold} results: F1={metrics['eval_f1']:.4f}, P={metrics['eval_precision']:.4f}, R={metrics['eval_recall']:.4f} (took {fold_elapsed:.0f}s)")

    # Save best model for this fold
    trainer.save_model(str(fold_output / "best"))

    # Free memory between folds
    del model, trainer
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()

cv_elapsed = time.time() - cv_start
print(f"\n{'='*60}")
print(f"5-Fold CV completed in {cv_elapsed:.0f}s ({cv_elapsed/60:.1f} min)")
print(f"{'='*60}")

In [ ]:
# Aggregate CV results
import pandas as pd

df = pd.DataFrame(fold_results)
display_cols = ["fold", "eval_precision", "eval_recall", "eval_f1", "best_epoch", "train_time_sec"]
print("\n5-Fold CV Results:")
print(df[display_cols].to_string(index=False))
print(f"\nMean F1: {df['eval_f1'].mean():.4f} ± {df['eval_f1'].std():.4f}")
print(f"Mean Precision: {df['eval_precision'].mean():.4f} ± {df['eval_precision'].std():.4f}")
print(f"Mean Recall: {df['eval_recall'].mean():.4f} ± {df['eval_recall'].std():.4f}")
print(f"Total CV time: {df['train_time_sec'].sum():.0f}s ({df['train_time_sec'].sum()/60:.1f} min)")

# Compute average best epoch from CV early stopping
best_epochs = []
for r in fold_results:
    ep = r.get("best_epoch")
    if ep is not None:
        try:
            best_epochs.append(int(ep))
        except (ValueError, TypeError):
            pass

if best_epochs:
    # batch_size=2, grad_accum=2 -> effective batch=4, steps/epoch = train_size / 2 / 2
    steps_per_epoch = len(load_bio_jsonl(DATA_DIR / "fold_0" / "train.jsonl")) // 2 // 2
    best_epoch_nums = [max(1, round(int(s) / steps_per_epoch)) for s in best_epochs]
    avg_best_epoch = max(1, int(round(np.mean(best_epoch_nums))))
    print(f"\nBest checkpoint steps: {best_epochs}")
    print(f"Steps/epoch: {steps_per_epoch}, Best epochs: {best_epoch_nums}")
    print(f"Average best epoch: {avg_best_epoch}")
else:
    avg_best_epoch = 10
    print(f"\nCould not determine best epochs, using default: {avg_best_epoch}")

## Train Final Model on All Data

After CV validation, train on all 120 records for the production model.

In [ ]:
# Train final model on all data
all_ds = load_bio_jsonl(DATA_DIR / "all.jsonl")
print(f"Training final model on {len(all_ds)} records for {avg_best_epoch} epochs")

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABEL_LIST),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "final"),
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,  # effective batch = 4
    num_train_epochs=avg_best_epoch,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=USE_FP16,
    use_mps_device=(DEVICE == "mps"),
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=all_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

final_start = time.time()
trainer.train()
final_elapsed = time.time() - final_start

trainer.save_model(str(OUTPUT_DIR / "final" / "best"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "final" / "best"))
print(f"Final model saved to {OUTPUT_DIR / 'final' / 'best'}")
print(f"Final training took {final_elapsed:.0f}s ({final_elapsed/60:.1f} min)")

## Evaluate Fine-Tuned Model on Gold Eval Split

Run the fine-tuned model on the 30 held-out eval records and compute P/R/F1/F2 per-type and per-language using the same span-overlap matching as `src/pii/eval_gold.py`.

In [ ]:
import re
from collections import Counter, defaultdict
from transformers import pipeline as hf_pipeline

FINAL_MODEL_PATH = OUTPUT_DIR / "final" / "best"
print(f"Loading fine-tuned model from {FINAL_MODEL_PATH}")

ner_pipe = hf_pipeline(
    task="token-classification",
    model=str(FINAL_MODEL_PATH),
    aggregation_strategy="simple",
    device="mps" if DEVICE == "mps" else (-1 if DEVICE == "cpu" else 0),
)

# Load gold eval records
GOLD_EVAL_PATH = DATA_DIR.parent / "gold_set" / "eval.jsonl"
if not GOLD_EVAL_PATH.exists():
    GOLD_EVAL_PATH = Path("data/prepared/gold_set/eval.jsonl")

gold_eval = []
with GOLD_EVAL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            gold_eval.append(json.loads(line))
print(f"Loaded {len(gold_eval)} eval records from {GOLD_EVAL_PATH}")

# Label mapping for NER output
BIO_PREFIX_RE = re.compile(r"^[BIES]-", flags=re.IGNORECASE)
NER_LABEL_MAP = {
    "PER": "NAME", "PERSON": "NAME",
    "LOC": "ADDRESS", "GPE": "ADDRESS",
    "ORG": "ORG",
    "NAME": "NAME", "PHONE": "PHONE", "EMAIL": "EMAIL",
    "ADDRESS": "ADDRESS", "ID": "ID",
}

# Run predictions
pred_records = []
for rec in gold_eval:
    text = rec.get("transcript", "")
    raw_ents = ner_pipe(text) if text.strip() else []
    spans = []
    for ent in raw_ents:
        raw_label = ent.get("entity_group") or ent.get("entity") or ""
        label = BIO_PREFIX_RE.sub("", raw_label).upper().strip()
        label = NER_LABEL_MAP.get(label, label)
        start, end = ent.get("start"), ent.get("end")
        if not isinstance(start, int) or not isinstance(end, int) or end <= start:
            continue
        spans.append({"start": start, "end": end, "type": label, "text": text[start:end], "score": float(ent.get("score", 0))})
    pred_records.append({"record_id": rec["record_id"], "spans": spans})

# Compute metrics using span-overlap matching (same logic as eval_gold.py)
def _char_overlap(s1s, s1e, s2s, s2e):
    return max(0, min(s1e, s2e) - max(s1s, s2s))

def _prf(tp, fp, fn, beta=2.0):
    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    fb = (1+beta**2)*p*r/(beta**2*p+r) if (beta**2*p+r) else 0
    return {"precision": p, "recall": r, "f1": f1, f"f{beta:.0f}": fb}

beta = 2.0
overlap_threshold = 0.5
total_tp = total_fp = total_fn = 0
type_counts = defaultdict(Counter)
lang_counts = defaultdict(Counter)

for gold_rec, pred_rec in zip(gold_eval, pred_records):
    entities = gold_rec.get("entities", [])
    pred_spans = pred_rec.get("spans", [])
    lang = gold_rec.get("language", "unknown")
    matched_gold, matched_pred = set(), set()

    for gi, ent in enumerate(entities):
        g_start, g_end = ent.get("start_char"), ent.get("end_char")
        g_type, g_text = ent.get("pii_type", ""), ent.get("text", "")
        if g_start is None or g_end is None:
            continue
        for pi, span in enumerate(pred_spans):
            if pi in matched_pred:
                continue
            if span.get("type", "") != g_type:
                continue
            p_start, p_end = span["start"], span["end"]
            overlap = _char_overlap(g_start, g_end, p_start, p_end)
            gold_len, pred_len = max(1, g_end - g_start), max(1, p_end - p_start)
            overlap_ratio = max(overlap/gold_len, overlap/pred_len)
            text_contained = (g_text in span.get("text", "")) or (span.get("text", "") in g_text)
            if overlap_ratio >= overlap_threshold or text_contained:
                matched_gold.add(gi)
                matched_pred.add(pi)
                type_counts[g_type]["tp"] += 1
                lang_counts[lang]["tp"] += 1
                break

    fn_ents = [gi for gi, e in enumerate(entities) if gi not in matched_gold and e.get("start_char") is not None]
    tp = len(matched_gold)
    fp = len(pred_spans) - len(matched_pred)
    fn = len(fn_ents)
    total_tp += tp; total_fp += fp; total_fn += fn
    lang_counts[lang]["fp"] += fp; lang_counts[lang]["fn"] += fn
    for gi in fn_ents:
        type_counts[entities[gi]["pii_type"]]["fn"] += 1
    for pi, span in enumerate(pred_spans):
        if pi not in matched_pred:
            type_counts[span["type"]]["fp"] += 1

# Display results
print(f"\n{'='*60}")
print("FINE-TUNED MODEL — Gold Eval Set Results (30 records)")
print(f"{'='*60}")

overall = _prf(total_tp, total_fp, total_fn, beta)
print(f"Overall: TP={total_tp} FP={total_fp} FN={total_fn}")
print(f"  Precision={overall['precision']:.3f}  Recall={overall['recall']:.3f}  F1={overall['f1']:.3f}  F2={overall['f2']:.3f}")

print(f"\nPer-Type:")
for pii_type in ["NAME", "PHONE", "EMAIL", "ADDRESS", "ID"]:
    c = type_counts[pii_type]
    m = _prf(c["tp"], c["fp"], c["fn"], beta)
    support = c["tp"] + c["fn"]
    print(f"  {pii_type:10s}  support={support:3d}  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  F2={m['f2']:.3f}")

print(f"\nPer-Language:")
for lang in sorted(lang_counts.keys()):
    c = lang_counts[lang]
    m = _prf(c["tp"], c["fp"], c["fn"], beta)
    print(f"  {lang:8s}  TP={c['tp']:3d}  FP={c['fp']:3d}  FN={c['fn']:3d}  P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  F2={m['f2']:.3f}")

In [ ]:
# Summary & Integration
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Base model:      {BASE_MODEL}")
print(f"Training data:   {len(load_bio_jsonl(DATA_DIR / 'all.jsonl'))} records (120 dev)")
print(f"Eval data:       {len(gold_eval)} records (30 held-out)")
print(f"CV mean F1:      {df['eval_f1'].mean():.4f} ± {df['eval_f1'].std():.4f}")
print(f"Best avg epoch:  {avg_best_epoch}")
print(f"Eval overall F1: {overall['f1']:.3f}")
print(f"Eval overall F2: {overall['f2']:.3f}")
print(f"Model saved at:  {(OUTPUT_DIR / 'final' / 'best').resolve()}")
print()
print("To use the fine-tuned model in the pipeline:")
print("  1. Update configs/pii.yaml:")
print(f'     pii.ner.model_name: "{OUTPUT_DIR / "final" / "best"}"')
print("  2. Re-run: python -m src.cli pii-eval-gold --config configs/pii_eval_gold.yaml")